# RSI-Jev v4.0-VL: typed questions about images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shanghua-Gao/RSI-Jev/blob/main/notebooks/rsi_jev_v4_vl_quickstart.ipynb)

Ask a yes/no or multiple-choice question about a picture and get a probability for every allowed answer, from one forward pass. Three pictures below: a photo from a URL, a chart drawn with matplotlib and an app screenshot.

* Cells that start with `# GPU` need a GPU runtime (*Runtime → Change runtime type → T4 GPU* or better). The other cells run on CPU.
* The weights load from the Hugging Face Hub (`shgao/rsi-jev-v4.0-vl-qwen3.5-2b`); `MODEL` also takes a local checkpoint directory.

In [ ]:
# Colab ships torchaudio built for its own torch. Installing rsi-jev upgrades torch (it needs
# torch >= 2.13), and torchaudio has no build for the new one, so remove it: RSI-Jev does not
# use it, and a stale torchaudio stops transformers from importing the model.
%pip uninstall -y -q torchaudio
%pip install -q "rsi-jev[fast,vision] @ git+https://github.com/Shanghua-Gao/RSI-Jev" matplotlib
# [vision] adds Pillow and torchvision, which image requests need. [fast] adds the fused
# DeltaNet kernels (CUDA only). If [fast] does not install on your runtime, drop it: the
# model runs without it, more slowly.

# A running session keeps the torch, torchaudio and transformers it already loaded, so on
# Colab this cell restarts the session once ("Your session crashed" is expected), then run
# the cells below. Running this cell again later does not restart it.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("/content/.rsijev_installed"):
    open("/content/.rsijev_installed", "w").close()
    os.kill(os.getpid(), 9)

In [ ]:
# GPU
from rsijev import Decider

MODEL = "shgao/rsi-jev-v4.0-vl-qwen3.5-2b"   # a local checkpoint directory works too
d = Decider(MODEL)       # a T4 has no native bf16; pass dtype="fp32" there if bf16 is slow
print(d)                 # device, dtype and the calibration the release ships

In [ ]:
def show(answers):
    """Print one line per question: the answer, its probability and the full distribution."""
    for key, a in answers.items():
        if a["type"] == "noul":
            print(f"{key:<13} p(yes) = {a['noul']:.2f}")
        else:
            dist = "  ".join(f"{k} {p:.2f}" for k, p in a["probabilities"].items())
            print(f"{key:<13} {a['choice']:<12} confidence {a['confidence']:.2f}   [{dist}]")

## 1. A photo from a URL

[Stop sign us.jpg](https://commons.wikimedia.org/wiki/File:Stop_sign_us.jpg) by Dori, public domain, via Wikimedia Commons.

In [ ]:
import io, urllib.request
from PIL import Image

URL = "https://upload.wikimedia.org/wikipedia/commons/d/de/Stop_sign_us.jpg"
req = urllib.request.Request(URL, headers={"User-Agent": "rsi-jev-quickstart (github.com/Shanghua-Gao/RSI-Jev)"})
photo = Image.open(io.BytesIO(urllib.request.urlopen(req).read())).convert("RGB")
photo.thumbnail((512, 512))
photo

In [ ]:
# GPU
show(d.decide("A frame from the car's front camera: <image>", {
    "sign": {"type": "choice", "instructions": "Which sign is in the picture?",
             "criteria": {"stop": "A stop sign.", "yield": "A yield or give-way sign.",
                          "speed_limit": "A speed limit sign.", "none": "No traffic sign."}},
    "must_stop": {"type": "noul", "instructions": "Must the car come to a full stop here?"},
}, images=[photo]))

## 2. A chart drawn with matplotlib

Sign-ups peak in April and fall in June, so the right answers are `Apr` and yes.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

months, signups = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"], [120, 135, 150, 172, 148, 90]
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=100)
ax.bar(months, signups, color="#4c72b0")
ax.set_title("New sign-ups per month")
ax.set_ylabel("sign-ups")
buf = io.BytesIO()
fig.savefig(buf, format="png", bbox_inches="tight")
plt.close(fig)
chart = Image.open(buf).convert("RGB")
chart

In [ ]:
# GPU
show(d.decide("Monthly report: <image>", {
    "peak": {"type": "choice", "instructions": "Which month has the most sign-ups?",
             "criteria": {m: None for m in months}},
    "dropped": {"type": "noul", "instructions": "Did sign-ups fall in the last month shown?"},
}, images=[chart]))

## 3. A screenshot

A notes app with a "Couldn't save" dialog, drawn by the package itself (`serve/demo_images.py`), so the answers are known: yes, and storage.

In [ ]:
from serve.demo_images import screenshot

shot, _ = screenshot()
shot

In [ ]:
# GPU
ticket = d.decide("A user attached this screenshot to a support ticket: <image>", {
    "error_dialog": {"type": "noul", "instructions": "Does the screenshot show an error dialog?"},
    "cause": {"type": "choice", "instructions": "What is the error about?",
              "criteria": {"storage": "Disk space or storage.", "network": "Connectivity or a server.",
                           "permission": "Access rights or sign-in.", "none": "There is no error."}},
}, images=[shot])
show(ticket)

## 4. Acting on the probability

The probabilities are calibrated: the release ships a fitted calibration (`d.calibration` names it) that divides each question's logits by one positive number, so the chosen answer never changes but a 0.9 is meant to be right about 9 times in 10. For a multiple-choice answer, `confidence` is (K·p_max − 1)/(K − 1), how far the top option sits above chance: 1 when all the mass is on one option, 0 when it is spread evenly.

That makes a threshold meaningful. Act on a confident answer, and send the unsure middle to a person:

In [ ]:
def route(p_yes, act_at=0.90, skip_at=0.10):
    if p_yes >= act_at:
        return "act"
    if p_yes <= skip_at:
        return "skip"
    return "ask a person"

for p in (0.97, 0.55, 0.04):
    print(f"p(yes) = {p:.2f} -> {route(p)}")

In [ ]:
# GPU
p = ticket["error_dialog"]["noul"]
print(f"error dialog: p(yes) = {p:.2f} -> {route(p)}")
# confidence 0.5 with four options means the top one has p = 0.625, halfway from chance (0.25) to certain
if ticket["cause"]["confidence"] < 0.5:
    print("cause unclear: attach the screenshot to the ticket for a person")
else:
    print("route the ticket to:", ticket["cause"]["choice"])